In [ ]:
import sphere_ref_lib as srl
import sphere_variables as sv

import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import xarray as xr

plt.rcParams["font.size"] = 20

#fp = srl.set_output_dir()
fp_nc = srl.set_output_dir_nc()
R, step, xs, d, Z0, _ = srl.set_basic_params()
_, _, target, apply_ref, fp, _ = sv.svv()
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = srl.get_default_param(target)
height = np.array([100,200,500,1000])

R2s =( height + (R_moon / 1000) )/ (R_moon/1000) # ganymedeにおいて100km, 200km, 500km, 1000kmを想定

R2s_make_nc = np.array(R2s)
srl.make_nc_alpha_ts(R2s_make_nc*1000, fp_nc, fn = "alpha_to_theta_forhist_ver3.nc")

plot_colors = ["red", "darkorange", "springgreen", "mediumblue", "fuchsia"]

In [ ]:
R2s

In [ ]:
for R2 in R2s:
    srl.make_nc_sphere(R2, step, xs, d, Z0, fp_nc)

In [ ]:
param_list, cor_list = [], []

for i in range(len(R2s)):

    R2 = R2s[i]

    theta_arr_r2, phi_arr_r2, p0s, p2s, p_hits, ref_dirs, amp_arr, alp_arr = srl.ref_rays_count_ref(R2, xs, d, Z0, R, y=0.0, print_info=False, reflectance=False, e1=e1)

    params = {
        "target" : target,
        "R2" : R2,
        "fp_nc" : fp_nc,
        "theta_arr_r2" : theta_arr_r2,
        "phi_arr_r2" : phi_arr_r2,
        "p0s" : p0s,
        "p2s" : p2s,
        "p_hits" : p_hits,
        "ref_dirs" : ref_dirs,
        "amp_arr" : amp_arr,
        "alp_arr" : alp_arr,
    }

    cors = [
        #("solid_angle", {}),
        #("solid_angle_field", {}),
        ("solid_angle_field_upd", {}),
        ("reflection_rate", {"apply_ref": "average", "fn": "alpha_to_theta_forhist_ver3.nc"}),
        ("vertical_direction", {}),
    ]
    param_list.append(params)
    cor_list.append(cors)

In [ ]:
sum_ray, sum_field, unit_inc = srl.calc_inc_field(1,20)

unit_inc

In [ ]:
sum_ray, sum_field, unit_inc = srl.calc_inc_field(1,20)

fig, ax = plt.subplots(figsize=(17,8))

ax.set_xticks(np.arange(0,181,10))
ax.grid()

test_counts = np.zeros(len(R2s))

for i in range(len(R2s)):

    R2 = R2s[i].round(3)
    params = param_list[i]
    cors = cor_list[i]

    ds_all = srl.load_nc_sphere(R2, fp_nc)

    heat_da = srl.make_thph_2dhist(ds_all)

    fin_da = srl.steve_correction_pipeline(heat_da, params, cors, print_info=False)

    phi_min, phi_max = 0.0, 90.0

    fin_da_090 = fin_da.where((fin_da["phi"]>=phi_min) & (fin_da["phi"]<=phi_max), drop=True)
    #th_1dcounts_090 = fin_da_090.sum(dim="phi")
    #th_1dcounts_090.name = "counts (sum over phi 0-90)"
    th_1dcounts_090 = fin_da_090.sum(dim="phi") / len(fin_da_090["phi"])
    th_1dcounts_090.name = "Ave counts (sum over phi 0-90)"

    
    ax.plot(th_1dcounts_090["theta"].values, th_1dcounts_090.values, lw=2, c=plot_colors[i], 
            label=f"R2={R2},height={height[i]}km")
    ax.set_title(f"Counts vs Theta (Counts over phi=0-90)")
    ax.set_xlabel("Theta (degrees)")
    ax.set_ylabel("Ave counts (Counts over phi 0-90)")
    #ax.set_xlim(90, 100)
    #ax.set_ylim(0,20000)

    test_counts[i] = th_1dcounts_090.values[0]

inc_counts = np.full_like(th_1dcounts_090.values, unit_inc)

ax.plot(th_1dcounts_090["theta"].values, inc_counts, lw=2, c="black", ls="--", label="incident beam")

print("\ncounts ratio : ", test_counts[0] / test_counts[-1])
print("\nDistance square ratio : ", ((R2s[-1]-0.5)/(R2s[0]-0.5))**2)

plt.legend()
plt.savefig(fp + "theta_counts_090_ganymede_oppotunity_forslide_all_apply.png")
plt.show()

In [ ]:
sum_ray, sum_field, unit_inc = srl.calc_inc_field(1,20)

fig, ax = plt.subplots(figsize=(17,8))

ax.set_xticks(np.arange(0,181,10))
ax.grid()

test_counts = np.zeros(len(R2s))

for i in range(len(R2s)):

    R2 = R2s[i].round(3)
    params = param_list[i]
    cors = cor_list[i]

    ds_all = srl.load_nc_sphere(R2, fp_nc)

    heat_da = srl.make_thph_2dhist(ds_all)

    fin_da = srl.steve_correction_pipeline(heat_da, params, cors, print_info=False)

    phi_min, phi_max = 0.0, 90.0

    fin_da_090 = fin_da.where((fin_da["phi"]>=phi_min) & (fin_da["phi"]<=phi_max), drop=True)
    #th_1dcounts_090 = fin_da_090.sum(dim="phi")
    #th_1dcounts_090.name = "counts (sum over phi 0-90)"
    th_1dcounts_090 = fin_da_090.sum(dim="phi") / len(fin_da_090["phi"])
    th_1dcounts_090.name = "Ave counts (sum over phi 0-90)"

    
    ax.plot(th_1dcounts_090["theta"].values, th_1dcounts_090.values / unit_inc, lw=2, c=plot_colors[i], 
            label=f"R2={R2},height={height[i]}km")
    ax.set_title(f"Ratio to incident beam vs Theta (Counts over phi=0-90)")
    ax.set_xlabel("Theta (degrees)")
    ax.set_ylabel("Ratio to incident beam (Ave of phi 0-90)")
    #ax.set_xlim(90, 100)
    #ax.set_ylim(0,20000)

    test_counts[i] = th_1dcounts_090.values[0]

print("\ncounts ratio : ", test_counts[0] / test_counts[-1])
print("\nDistance square ratio : ", ((R2s[-1]-0.5)/(R2s[0]-0.5))**2)

plt.legend()
plt.savefig(fp + "theta_ratio_090_ganymede_oppotunity_forslide_all_apply.png")
plt.show()

In [ ]:
a = np.array([1,2])
b = np.array([4,6])
np.linalg.norm(b-a, axis=0)

In [ ]:
a = np.array([1,2,3])
b = np.array([a*2, a*3])
b